#### Import Packages

In [2]:
import pandas as pd
import numpy as np

#### Import DataFrames from Scraping

In [3]:
from scrape_script import StriveDF, XrdDF


c:\Users\brock\OneDrive\Desktop\BYU\Semester 7\STAT_386\Data_acquisition_blog\STAT-386-Data-Acquisition-Code\scrape_script.py:7: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 7 of the file c:\Users\brock\OneDrive\Desktop\BYU\Semester 7\STAT_386\Data_acquisition_blog\STAT-386-Data-Acquisition-Code\scrape_script.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  soup = BeautifulSoup(r.text)
c:\Users\brock\OneDrive\Desktop\BYU\Semester 7\STAT_386\Data_acquisition_blog\STAT-386-Data-Acquisition-Code\scrape_script.py:18: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parse

#### Set Up Data Combining

In [4]:
def blank_data_dict():
    data_dict = {
        "name": [],
        "5P": [],
        "5K": [],
        "c.S": [],
        "f.S": [],
        "5H": [],
        "5D": [],
        "2P": [],
        "2K": [],
        "2S": [],
        "2H": [],
        "2D": [],
        "6P": []
    }
    return data_dict

#### Extract Move Properties

In [5]:
def create_property_given_game_df(game, property):
    data_dict = blank_data_dict()
    
    if game == "Strive":
        data = StriveDF
    elif game == "Xrd":
        data = XrdDF

    for item in data_dict:
        if item == "name":
            data_dict[item] = list(data["name"])
        else:
            data_dict[item] = find_given_move_property(data, [item], property)
    
    new_DF = pd.DataFrame(data_dict)
    return new_DF


#### Compile Move Properties

In [6]:
def find_given_move_property(df, property):
    result_property = []
    for column in df.columns:
        if column != "name":
            given_attack_property = []
            index = property_index_dict[property]
            for attack in df[column]:
                try:
                    given_attack_property.append(attack[index])
                except TypeError:
                    given_attack_property.append(None)
            result_property.append(given_attack_property)
    return result_property

def compile_move_properties(property):
    GGST_property = find_given_move_property(StriveDF, property)
    zip_strive_property = list(zip(
        GGST_property[0], 
        GGST_property[1], 
        GGST_property[2], 
        GGST_property[3], 
        GGST_property[4], 
        GGST_property[5], 
        GGST_property[6], 
        GGST_property[7], 
        GGST_property[8], 
        GGST_property[9], 
        GGST_property[10], 
        GGST_property[11],
        GGST_property[12]
    ))


    Xrd_property = find_given_move_property(XrdDF, property)
    zip_xrd_property = list(zip(
        Xrd_property[0], 
        Xrd_property[1], 
        Xrd_property[2], 
        Xrd_property[3], 
        Xrd_property[4], 
        Xrd_property[5], 
        Xrd_property[6], 
        Xrd_property[7], 
        Xrd_property[8], 
        Xrd_property[9], 
        Xrd_property[10], 
        Xrd_property[11]
    ))

    property = []
    for character_data in zip_strive_property:
        for character_move in character_data:
            property.append(character_move)
    for character_data in zip_xrd_property:
        for character_move in character_data:
            property.append(character_move)
    return property


#### Create Combined DataFrame

In [40]:
data_dict = {
    "attack": [],
    "game": [],
    "attack_type": [],
    "guard": [],
    "startup": [],
    "recovery": [],
    "on_block": []
 
}

attacks = []
game = []
attack_type = []
game_indicator = "Strive"
for df in [StriveDF, XrdDF]:
    for character in df["name"]:
        for column in df.columns:
            if column != "name":
                attacks.append(f"{character}{column} ({game_indicator})")
                game.append(game_indicator)
                attack_type.append(column)
    game_indicator = "Xrd"

property_index_dict = {
    "guard": 0,
    "startup": 1,
    "recovery": 3,
    "on_block": 4
}

print(len(attacks))
data_dict["attack"] = attacks
data_dict["game"] = game
data_dict["attack_type"] = attack_type
data_dict["guard"] = compile_move_properties("guard")
data_dict["startup"] = compile_move_properties("startup")
data_dict["recovery"] = compile_move_properties("recovery")
data_dict["on_block"] = compile_move_properties("on_block")

final_df = pd.DataFrame(data_dict)
final_df.head()

742


,attack,game,attack_type,guard,startup,recovery,on_block
0,Robo-Ky5P (Strive),Strive,5P,All,5,9,-2
1,Robo-Ky5K (Strive),Strive,5K,All,7,11,-2
2,Robo-Kyc.S (Strive),Strive,c.S,All,7,10,+1
3,Robo-Kyf.S (Strive),Strive,f.S,All,10,20,-9
4,Robo-Ky5H (Strive),Strive,5H,All,13,24,-8


#### Clean Data

In [ ]:
final_df_cleaned = final_df.copy()
final_df_cleaned.dropna()

final_df_cleaned["guard"] = (
    final_df_cleaned["guard"]
    .str.replace(r"Mid", "All", regex = True)
    .str.replace(r"All.*", "All", regex = True)
    .str.replace(r"High.*", "High", regex = True)
    .str.replace(r"Low.*", "Low", regex = True)
)
final_df_cleaned["startup"] = (
    final_df_cleaned["startup"]
    .str.replace(r"\{*\[.*", "", regex = True)
    .str.replace(r"[^\d+\+]", "", regex = True)
    .astype(float)
)

def recovery_clean(lst):
  if type(lst) == list:
    if len(lst) == 1:
        return float(lst[0])
    elif len(lst) > 1:
        return float(float(lst[0]) + float(lst[1]))

final_df_cleaned["recovery"] = (
    final_df_cleaned["recovery"]
    .str.replace(r"\[.*", "", regex = True)
    .str.replace(r"[^\d+\+]", "", regex = True)
    .str.split("+")
    .apply(recovery_clean)
    .astype(float)
)
final_df_cleaned["on_block"] = (
    final_df_cleaned["on_block"]
    .str.replace(r"\[.*", "", regex = True)
    .str.replace(r"~.*", "", regex = True)
    .str.replace(r"[^\d+\+\-]", "", regex = True)
    .astype(float)
)
final_df_cleaned["plus?"] = (
    final_df_cleaned["on_block"] > 0
)

print(f"Final Length: {len(final_df_cleaned)}")
final_df_cleaned.head(10)

742


,attack,game,attack_type,guard,startup,recovery,on_block,plus?
0,Robo-Ky5P (Strive),Strive,5P,All,5.0,9.0,-2.0,False
1,Robo-Ky5K (Strive),Strive,5K,All,7.0,11.0,-2.0,False
2,Robo-Kyc.S (Strive),Strive,c.S,All,7.0,10.0,1.0,True
3,Robo-Kyf.S (Strive),Strive,f.S,All,10.0,20.0,-9.0,False
4,Robo-Ky5H (Strive),Strive,5H,All,13.0,24.0,-8.0,False
5,Robo-Ky5D (Strive),Strive,5D,High,20.0,25.0,-15.0,False
6,Robo-Ky5[D] (Strive),Strive,5[D],High,28.0,25.0,-10.0,False
7,Robo-Ky2P (Strive),Strive,2P,All,5.0,9.0,-2.0,False
8,Robo-Ky2K (Strive),Strive,2K,Low,6.0,11.0,-2.0,False
9,Robo-Ky2S (Strive),Strive,2S,Low,12.0,22.0,-10.0,False


### Run EDA

##### Analyze General Trends Between Games

In [17]:
final_df_cleaned.groupby("game").agg(
    average_startup = ("startup", "mean"),
    average_recovery = ("recovery", "mean"),
    average_on_block = ("on_block", "mean"),
    startup_variance = ("startup", "var"),
    recovery_variance = ("recovery", "var"),
    on_block_variance = ("on_block", "var"),
    fastest_move = ("startup", "min"),
    slowest_move = ("startup", "max"),
    plus_count = ("plus?", "sum"),
    plus_average = ("plus?", "mean"),
    num_moves = ("game", "count")
)


,average_startup,average_recovery,average_on_block,startup_variance,recovery_variance,on_block_variance,fastest_move,slowest_move,plus_count,plus_average,num_moves
game,,,,,,,,,,,
Strive,11.561364,17.552273,-7.245455,40.848162,50.575849,32.345082,5.0,28.0,37,0.08371,442
Xrd,9.896667,14.213333,-4.693333,34.026076,47.258685,37.116343,3.0,29.0,57,0.19000,300


##### Analyze Trends in Attack Types

In [42]:
final_df_cleaned.groupby(["attack_type", "game"]).agg(
    average_startup = ("startup", "mean"),
    average_recovery = ("recovery", "mean"),
    average_on_block = ("on_block", "mean"),
    startup_variance = ("startup", "var"),
    recovery_variance = ("recovery", "var"),
    on_block_variance = ("on_block", "var"),
    fastest_move = ("startup", "min"),
    slowest_move = ("startup", "max"),
    plus_count = ("plus?", "sum"),
    plus_mean = ("plus?", "mean") # Among given attack types, what percentage of instances are plus?
)

average_startup  average_recovery  average_on_block  \
attack_type game                                                          
2D          Strive        11.235294         18.500000         -8.794118   
            Xrd            9.200000         16.920000         -8.720000   
2H          Strive        14.727273         25.575758        -12.242424   
            Xrd           12.680000         22.040000         -9.280000   
2K          Strive         6.029412         11.147059         -2.823529   
            Xrd            6.080000          7.600000         -0.360000   
2P          Strive         5.529412          9.705882         -2.470588   
            Xrd            5.440000          7.000000         -0.040000   
2S          Strive        10.823529         18.558824         -8.470588   
            Xrd            9.440000         15.840000         -5.760000   
5D          Strive        20.000000         25.176471        -15.000000   
            Xrd           26.080000         15.160000         -5.560000   
5H          Strive        13.212121         22.818182         -8.424242   
            Xrd           11.440000         19.000000         -4.920000   
5K          Strive         7.147059         11.058824         -3.441176   
            Xrd            6.760000         10.640000         -3.560000   
5P          Strive         5.470588          8.735294         -1.970588   
            Xrd            5.080000          7.320000         -0.560000   
5[D]        Strive        28.000000         25.176471        -10.000000   
6P          Strive        10.205882         22.235294        -13.441176   
            Xrd           10.840000         18.440000         -8.320000   
c.S         Strive         7.176471         10.558824          1.264706   
            Xrd            6.520000         12.640000         -1.240000   
f.S         Strive        10.882353         19.323529         -8.558824   
            Xrd            9.200000         17.960000         -8.000000   

                    startup_variance  recovery_variance  on_block_variance  \
attack_type game                                                             
2D          Strive          2.670232           9.772727          10.168449   
            Xrd             4.500000          53.326667          71.626667   
2H          Strive         13.142045          62.001894          44.564394   
            Xrd            13.976667          35.873333          50.293333   
2K          Strive          0.271836           6.129234           3.907308   
            Xrd             0.910000           3.916667           4.490000   
2P          Strive          1.347594           6.153298           5.529412   
            Xrd             2.590000           7.833333           7.706667   
2S          Strive          1.907308          11.647950          14.499109   
            Xrd             7.006667          70.140000          73.606667   
5D          Strive          0.000000           3.786096           0.000000   
            Xrd             1.826667          28.556667          23.090000   
5H          Strive          4.297348          18.840909          14.251894   
            Xrd             5.006667          27.333333          27.910000   
5K          Strive          1.159537          10.178253           6.981283   
            Xrd             1.773333           9.240000           9.756667   
5P          Strive          0.620321           5.655080           4.574866   
            Xrd             1.243333           8.726667           8.506667   
5[D]        Strive          0.000000           3.786096           0.000000   
6P          Strive          1.501783           6.609626           6.617647   
            Xrd            12.890000          15.923333          19.060000   
c.S         Strive          0.149733           1.284314           1.291444   
            Xrd             1.593333          15.323333          10.940000   
f.S         Strive          2.167558        

##### Analyze Trends in Highs/Lows/All

In [43]:
final_df_cleaned.groupby(["guard", "game"]).agg(
    average_startup = ("startup", "mean"),
    average_recovery = ("recovery", "mean"),
    average_on_block = ("on_block", "mean"),
    startup_variance = ("startup", "var"),
    recovery_variance = ("recovery", "var"),
    on_block_variance = ("on_block", "var"),
    fastest_move = ("startup", "min"),
    slowest_move = ("startup", "max"),
    plus_count = ("plus?", "sum")
)

average_startup  average_recovery  average_on_block  \
guard game                                                          
All   Strive         9.205128         16.091575         -6.073260   
      Xrd            8.388350         14.650485         -4.810680   
High  Strive        24.000000         25.176471        -12.500000   
      Xrd           25.961538         15.423077         -5.807692   
Low   Strive         9.515152         16.343434         -6.868687   
      Xrd            8.323529         12.426471         -3.911765   

              startup_variance  recovery_variance  on_block_variance  \
guard game                                                             
All   Strive         12.869532          50.237907          34.112260   
      Xrd            10.882595          51.272366          43.793251   
High  Strive         16.238806           3.729587           6.343284   
      Xrd             2.118462          29.213846          23.761538   
Low   Strive          9.109462          36.819625          22.747887   
      Xrd             8.102722          38.725856          21.633889   

              fastest_move  slowest_move  plus_count  
guard game                                            
All   Strive           5.0          27.0          37  
      Xrd              3.0          25.0          42  
High  Strive          20.0          28.0           0  
      Xrd             23.0          29.0           3  
Low   Strive           5.0          17.0           0  
      Xrd              4.0          17.0          12

##### Analyze Trends in Advantageous on Block Moves

In [44]:
final_df_cleaned[final_df_cleaned["plus?"] == True].groupby("game").agg(
    average_startup = ("startup", "mean"),
    average_recovery = ("recovery", "mean"),
    average_on_block = ("on_block", "mean"),
    fastest_move = ("startup", "min"),
    slowest_move = ("startup", "max"),
    number_of = ("attack_type", "count")
)

,average_startup,average_recovery,average_on_block,fastest_move,slowest_move,number_of
game,,,,,,
Strive,7.702703,10.162162,1.756757,5.0,27.0,37
Xrd,7.719298,7.561404,2.052632,3.0,27.0,57


##### Analyze Trends in Advantageous on Block Moves of the Same Attack Type

In [ ]:
plus_attack_distribution = final_df_cleaned[final_df_cleaned["plus?"] == True].groupby(["game", "attack_type"]).agg(
    average_startup = ("startup", "mean"),
    average_recovery = ("recovery", "mean"),
    average_on_block = ("on_block", "mean"),
    fastest_move = ("startup", "min"),
    slowest_move = ("startup", "max"),
    number_of = ("attack_type", "count")
)
plus_attack_distribution

average_startup  average_recovery  average_on_block  \
game   attack_type                                                        
Strive 2H                 27.000000          9.000000          6.000000   
       2P                  5.000000          6.000000          3.000000   
       5P                  6.000000          6.000000          2.000000   
       c.S                 7.096774         10.419355          1.483871   
       f.S                10.500000         13.000000          3.000000   
Xrd    2H                 11.000000         11.000000          6.000000   
       2K                  6.500000          6.750000          2.000000   
       2P                  4.846154          5.692308          1.769231   
       2S                 10.000000          8.000000          3.000000   
       5D                 26.666667          9.666667          1.333333   
       5H                 12.250000         12.500000          2.500000   
       5K                  6.500000          6.250000          1.750000   
       5P                  4.800000          6.100000          1.500000   
       6P                  9.000000         12.000000          2.000000   
       c.S                 6.555556          9.444444          2.111111   
       f.S                13.000000          6.000000          6.000000   

                    fastest_move  slowest_move  number_of  
game   attack_type                                         
Strive 2H                   27.0          27.0          1  
       2P                    5.0           5.0          1  
       5P                    5.0           7.0          2  
       c.S                   7.0           8.0         31  
       f.S                  10.0          11.0          2  
Xrd    2H                   11.0          11.0          1  
       2K                    5.0           8.0          8  
       2P                    4.0           7.0         13  
       2S                   10.0          10.0          3  
       5D                   26.0          27.0          3  
       5H                   10.0          13.0          4  
       5K                    4.0           8.0          4  
       5P                    3.0           6.0         10  
       6P                    9.0           9.0          1  
       c.S                   4.0           8.0          9  
       f.S                  13.0          13.0          1

##### Analyze What Percentage of Attack Types are Advantageous on Block in Strive

In [37]:
plus_attack_distribution.iloc[0:5,-1]/37

game    attack_type
Strive  2H             0.027027
        2P             0.027027
        5P             0.054054
        c.S            0.837838
        f.S            0.054054
Name: number_of, dtype: float64

##### Analyze What Percentage of Attack Types are Advantageous on Block in Xrd

In [38]:
plus_attack_distribution.iloc[5:,-1]/37

game  attack_type
Xrd   2H             0.027027
      2K             0.216216
      2P             0.351351
      2S             0.081081
      5D             0.081081
      5H             0.108108
      5K             0.108108
      5P             0.270270
      6P             0.027027
      c.S            0.243243
      f.S            0.027027
Name: number_of, dtype: float64